# Scraping y procesamiento - Servicios básicos en la Región de Los Ríos

Extrae de la Biblioteca del Congreso Nacional (BCN) el indicador de carencia de
servicios básicos (RSH) para las 12 comunas de la Región de Los Ríos, en 2017 y
2024, y lo deja listo para análisis en `indicadores_vivienda.csv`.

Pueden saltarse el scraping y ejecutar 02_visualizaciones.ipynb: `bcn_servicios_basicos.csv` e `indicadores_vivienda.csv` ya vienen generados en el repositorio.

In [1]:
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
COMUNAS_LOS_RIOS = {
    "Valdivia": "14101",
    "Corral": "14102",
    "Lanco": "14103",
    "Los Lagos": "14104",
    "Máfil": "14105",
    "Mariquina": "14106",
    "Paillaco": "14107",
    "Panguipulli": "14108",
    "La Unión": "14201",
    "Futrono": "14202",
    "Lago Ranco": "14203",
    "Río Bueno": "14204",
}

YEARS = [2017, 2024]
BASE_URL = "https://www.bcn.cl/siit/reportescomunales/comunas_v.html?anno={year}&idcom={cut}"
HEADERS = {"User-Agent": "Mozilla/5.0"}

ARCHIVO_BCN = Path("bcn_servicios_basicos.csv")
ARCHIVO_INDICADORES = Path("indicadores_vivienda.csv")

## Funciones de extracción

Cada reporte comunal de la BCN trae una tabla con el porcentaje de personas en
hogares carentes de servicios básicos. `limpiar_valor` normaliza el formato del número (coma decimal, símbolo `%`) y `extraer_carencia_servicios`
ubica esa tabla dentro del HTML y saca el dato de la fila de la comuna.

In [3]:
def limpiar_valor(valor_texto):
    try:
        valor = valor_texto.strip().replace("%", "").strip().replace(",", ".")
        valor = "".join(c for c in valor if c.isdigit() or c == ".")
        if valor and valor != ".":
            return float(valor)
    except (ValueError, AttributeError):
        pass
    return None


def extraer_carencia_servicios(soup, nombre_comuna):
    registros = []
    try:
        titulo = soup.find(
            lambda tag: tag.name in ["h4", "h5", "h6"]
            and "servicios básicos" in tag.get_text().lower()
            and "hacinado" in tag.get_text().lower()
        )
        if not titulo:
            return registros

        tabla = titulo.find_next("table")
        if not tabla:
            return registros

        for fila in tabla.find_all("tr"):
            celdas = fila.find_all(["td", "th"])
            if len(celdas) < 3:
                continue

            unidad_territorial = celdas[0].get_text(strip=True).lower()
            if "comuna" not in unidad_territorial and nombre_comuna.lower() not in unidad_territorial:
                continue

            # El sitio cambió de formato entre años: con 4+ columnas el valor
            # está en el índice 2, con el diseño antiguo (3 columnas) en el 1.
            indice_valor = 2 if len(celdas) >= 4 else 1
            valor = limpiar_valor(celdas[indice_valor].get_text(strip=True))
            if valor is not None:
                registros.append({"comuna": nombre_comuna, "tipo_servicio": "carencia_servicios_basicos", "valor": valor})
    except Exception as e:
        print(f"Error extrayendo datos para {nombre_comuna}: {e}")

    return registros

## Scraper

Recorre las 12 comunas para 2017 y 2024 (24 páginas en total), respetando un
segundo de espera entre solicitudes.

In [4]:
def scraper_bcn():
    print(f"Comunas a procesar: {len(COMUNAS_LOS_RIOS)}\n")
    registros_totales = []

    for year in YEARS:
        for nombre_comuna, cut in COMUNAS_LOS_RIOS.items():
            try:
                url = BASE_URL.format(year=year, cut=cut)
                print(f"Scrapeando: {nombre_comuna} (año {year})...", end=" ")

                response = requests.get(url, headers=HEADERS, timeout=15)
                if response.status_code == 404:
                    print("Error 404")
                    continue

                response.raise_for_status()
                response.encoding = "utf-8"
                soup = BeautifulSoup(response.text, "html.parser")

                datos_comuna = extraer_carencia_servicios(soup, nombre_comuna)
                if datos_comuna:
                    for r in datos_comuna:
                        r["fuente"] = f"BCN (RSH {year})"
                        r["año"] = int(year)
                    registros_totales.extend(datos_comuna)
                    print(f"OK: {datos_comuna[0]['valor']}% de carencia")
                else:
                    print("estructura de tabla no encontrada")

                time.sleep(1)

            except Exception as e:
                print(f"Error: {e}")

    if not registros_totales:
        print("\nNo se obtuvieron datos de la BCN para los años solicitados")
        return None

    df = pd.DataFrame(registros_totales)
    df.to_csv(ARCHIVO_BCN, index=False, encoding="utf-8-sig")
    print(f"\nGuardado: {len(df)} registros en {ARCHIVO_BCN.name}")
    return df

## Ejecutar (o reutilizar el CSV ya scrapeado)

Pon `FORZAR_SCRAPING = True` si quieren volver a consultar el sitio de la BCN;
de lo contrario se reutiliza `bcn_servicios_basicos.csv`.

In [ ]:
FORZAR_SCRAPING = False

if FORZAR_SCRAPING or not ARCHIVO_BCN.exists():
    df_bcn = scraper_bcn()
else:
    df_bcn = pd.read_csv(ARCHIVO_BCN, encoding="utf-8-sig")
    print(f"Reutilizando {ARCHIVO_BCN.name} ({len(df_bcn)} registros)")

df_bcn.head()

Comunas a procesar: 12

Scrapeando: Valdivia (año 2017)... OK: 11.3% de carencia
Scrapeando: Corral (año 2017)... OK: 59.1% de carencia
Scrapeando: Lanco (año 2017)... OK: 19.3% de carencia
Scrapeando: Los Lagos (año 2017)... OK: 47.7% de carencia
Scrapeando: Máfil (año 2017)... OK: 31.8% de carencia
Scrapeando: Mariquina (año 2017)... OK: 35.7% de carencia
Scrapeando: Paillaco (año 2017)... OK: 21.9% de carencia
Scrapeando: Panguipulli (año 2017)... OK: 59.6% de carencia
Scrapeando: La Unión (año 2017)... OK: 30.9% de carencia
Scrapeando: Futrono (año 2017)... OK: 40.6% de carencia
Scrapeando: Lago Ranco (año 2017)... OK: 41.5% de carencia
Scrapeando: Río Bueno (año 2017)... OK: 42.3% de carencia
Scrapeando: Valdivia (año 2024)... OK: 11.6% de carencia
Scrapeando: Corral (año 2024)... OK: 52.6% de carencia
Scrapeando: Lanco (año 2024)... OK: 20.6% de carencia
Scrapeando: Los Lagos (año 2024)... OK: 45.3% de carencia
Scrapeando: Máfil (año 2024)... OK: 31.9% de carencia
Scrapeando: Mar

,comuna,tipo_servicio,valor,fuente,año
0,Valdivia,carencia_servicios_basicos,11.3,BCN (RSH 2017),2017
1,Corral,carencia_servicios_basicos,59.1,BCN (RSH 2017),2017
2,Lanco,carencia_servicios_basicos,19.3,BCN (RSH 2017),2017
3,Los Lagos,carencia_servicios_basicos,47.7,BCN (RSH 2017),2017
4,Máfil,carencia_servicios_basicos,31.8,BCN (RSH 2017),2017


## Procesamiento

Se valida el valor numérico y se pivotea para tener una fila por comuna con una
columna por indicador y año (`carencia_servicios_basicos_2017`,
`carencia_servicios_basicos_2024`).

In [6]:
df = df_bcn.copy()
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
df = df.dropna(subset=["valor"])

df_pivot = df.pivot_table(
    index="comuna",
    columns=["tipo_servicio", "año"],
    values="valor",
    aggfunc="mean",
)
df_pivot.columns = [f"{servicio}_{anio}" for servicio, anio in df_pivot.columns]
df_pivot = df_pivot.reset_index()

df_pivot.to_csv(ARCHIVO_INDICADORES, index=False, encoding="utf-8-sig")
print(f"Guardado: {ARCHIVO_INDICADORES.name}")
df_pivot

Guardado: indicadores_vivienda.csv


,comuna,carencia_servicios_basicos_2017,carencia_servicios_basicos_2024
0,Corral,59.1,52.6
1,Futrono,40.6,37.4
2,La Unión,30.9,27.4
3,Lago Ranco,41.5,34.3
4,Lanco,19.3,20.6
5,Los Lagos,47.7,45.3
6,Mariquina,35.7,25.9
7,Máfil,31.8,31.9
8,Paillaco,21.9,24.8
9,Panguipulli,59.6,53.3
